# Saving class probabilities

By default, a segmentation model writes one class per pixel: the class with the highest probability. With `save_probabilities=True`, the class probabilities (after softmax) are also written to a separate GeoTIFF, with one band per class. The class prediction itself stays the same.

The probabilities tell you how sure the model is about each pixel. You can use them to find uncertain areas, leave uncertain pixels unclassified, choose your own decision rule, or combine the predictions of several models.

In [ ]:
from pathlib import Path

import numpy as np
import rasterio as rst
from lightning import seed_everything
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import ModelCheckpoint
from matplotlib import pyplot as plt

from eotorch.data import (
    RasterDataModule,
    SegmentationTask,
    get_segmentation_dataset,
    splits,
)
from eotorch.plot import plot_class_raster
from eotorch.processing.normalize import band_statistics

# class mapping, used for plotting and for the band names of the probabilities file
class_mapping = {
    1: "Optically deep water",
    2: "Sand",
    3: "Coral",
    4: "SAV",
}
DATASET_DIR = Path("tests/data")

### Train a model

We quickly train a model on the test data. The steps are explained in detail in [segmentation.ipynb](segmentation.ipynb).

If you already have a checkpoint of a segmentation model, set `checkpoint` to its path and skip this section.

In [ ]:
seed_everything(43, workers=False)

ds = get_segmentation_dataset(
    images_dir=DATASET_DIR,
    labels_dir=DATASET_DIR,
    image_glob="image_*.tif",
    label_glob="label_*.tif",
    all_image_bands=("red", "green", "blue"),
    rgb_bands=("red", "green", "blue"),
    class_mapping=class_mapping,
    reduce_zero_label=True,
)
train_ds, val_ds = splits.file_wise_split(
    dataset=ds, val_img_files=DATASET_DIR / "image_3.tif"
)
mean, std = band_statistics(train_ds)
module = RasterDataModule(
    train_dataset=train_ds,
    val_dataset=val_ds,
    batch_size=16,
    patch_size=64,
    mean=mean,
    std=std,
)
task = SegmentationTask(
    model="deepresunet",
    in_channels=3,
    num_classes=len(class_mapping),
    ignore_index=-1,  # because reduce_zero_label=True
)
trainer = Trainer(
    callbacks=[ModelCheckpoint(monitor="val/F1_Score", mode="max")],
    max_epochs=30,
)
trainer.fit(model=task, datamodule=module)
checkpoint = trainer.checkpoint_callback.best_model_path

### Predict classes and probabilities

With `save_probabilities=True`, two files are written:

- `image_3_pred.tif`: the predicted class of each pixel, as without the option
- `image_3_pred_probabilities.tif`: the probability of each class, as float32, one band per class

The bands are named after the classes in the class mapping saved in the checkpoint. Since the model was trained with `reduce_zero_label=True`, band 1 holds the probability of class 1, band 2 of class 2, and so on. Pixels without a prediction are NaN.

Saving the probabilities works with and without `blend`. With `blend=True`, the probabilities of overlapping patches are averaged, and the classes are derived from the averaged probabilities.

In [ ]:
pred_path = SegmentationTask.predict_on_tif_file(
    tif_file_path=DATASET_DIR / "image_3.tif",
    checkpoint_path=checkpoint,
    out_file_path="predictions/image_3_pred.tif",
    blend=True,
    save_probabilities=True,
)
prob_path = pred_path.with_name(f"{pred_path.stem}_probabilities.tif")
print(pred_path)
print(prob_path)

### Inspect the probabilities

The band names are stored as band descriptions, which GIS software such as QGIS shows as the band names. The probabilities of each pixel add up to 1.

In [ ]:
with rst.open(prob_path) as src:
    band_names = src.descriptions
    probs = src.read()
    print(f"Bands: {band_names}")
    print(f"Data type: {src.dtypes[0]}, nodata: {src.nodata}")

with rst.open(pred_path) as src:
    classes = src.read(1)

valid = ~np.isnan(probs).any(axis=0)
print(f"Probabilities add up to 1: {np.allclose(probs[:, valid].sum(axis=0), 1, atol=1e-5)}")
# The predicted class is the one with the highest probability (+1 for reduce_zero_label)
print(f"Classes match the probabilities: {(probs.argmax(axis=0)[valid] + 1 == classes[valid]).all()}")

Plot the probability of each class. All maps share the same 0 to 1 scale, so they can be compared directly.

In [ ]:
fig, axes = plt.subplots(1, len(band_names), figsize=(4 * len(band_names), 4), layout="constrained")
for ax, name, band in zip(axes, band_names, probs):
    im = ax.imshow(band, cmap="Blues", vmin=0, vmax=1)
    ax.set_title(name)
    ax.set_axis_off()
fig.colorbar(im, ax=axes, shrink=0.8, label="Probability")
plt.show()

### How confident is the model?

The highest probability of a pixel shows how confident the model is about its predicted class. Low values often appear at the boundaries between classes, and in areas that look different from the training data.

In [ ]:
confidence = np.nanmax(probs, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), layout="constrained")
plot_class_raster(pred_path, ax=axes[0], class_mapping=class_mapping)
axes[0].set_title("Predicted class")
axes[0].set_axis_off()
im = axes[1].imshow(confidence, cmap="Blues", vmin=0, vmax=1)
axes[1].set_title("Confidence (highest probability)")
axes[1].set_axis_off()
fig.colorbar(im, ax=axes[1], shrink=0.8, label="Probability")
plt.show()

### Example: leave uncertain pixels unclassified

Pixels whose highest probability is below a threshold are set to the nodata value of the class prediction (0), and written to a new file.

How confident a model is depends on how well it is trained. The quickly trained model above is rarely confident, so here the threshold leaves the 20 % least confident pixels unclassified. For a well-trained model, you would typically use a fixed probability instead, e.g. `THRESHOLD = 0.6`.

In [ ]:
THRESHOLD = np.nanpercentile(confidence, 20)
print(f"Threshold: {THRESHOLD:.2f}")

with rst.open(pred_path) as src:
    profile = src.profile
confident_classes = np.where(confidence >= THRESHOLD, classes, profile["nodata"])

confident_path = pred_path.with_name(f"{pred_path.stem}_confident.tif")
with rst.open(confident_path, "w", **profile) as dst:
    dst.write(confident_classes.astype(profile["dtype"]), 1)

unclassified = (confidence[valid] < THRESHOLD).mean()
print(f"{unclassified:.1%} of the pixels are left unclassified")
plot_class_raster(confident_path, class_mapping=class_mapping)

### Using your own prediction function

`predict_on_tif_generic` has the same option. The probabilities file holds the output of `prediction_func`, before it is converted to classes by `postprocess`:

```python
from eotorch.inference import predict_on_tif_generic

predict_on_tif_generic(
    tif_file_path="image.tif",
    prediction_func=my_model_probabilities,  # returns (batch, classes, height, width)
    postprocess=lambda probs: probs.argmax(axis=-3).astype("uint8"),
    save_probabilities=True,
    probability_band_names=["Water", "Sand", "Coral", "SAV"],
)
```